In [1]:
!pip install -q openai-whisper
!pip install -q webrtcvad
!pip install -q pydub
!pip install -q transformers rouge-score datasets



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import files
uploaded = files.upload()

audio_path = list(uploaded.keys())[0]
print("Using audio:", audio_path)


Saving ya.mp3 to ya.mp3
Using audio: ya.mp3


In [3]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(audio_path)

text = result["text"]
print("TRANSCRIPT:\n", text)


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 131MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


TRANSCRIPT:
  Real self-confidence doesn't come from shouting affirmations in the mirror. Real self-confidence comes from giving the world irrefutable proof you are who you say you are. I fucking love that.


In [4]:
import webrtcvad
import wave
import contextlib
from pydub import AudioSegment

audio = AudioSegment.from_file(audio_path)
audio = audio.set_channels(1).set_frame_rate(16000)

audio.export("temp.wav", format="wav")

vad = webrtcvad.Vad(2)

with wave.open("temp.wav", "rb") as wf:
    sample_rate = wf.getframerate()
    frame_duration = 30
    frame_size = int(sample_rate * frame_duration / 1000) * 2
    audio_bytes = wf.readframes(wf.getnframes())

segments = []
start = 0
speaker = 1

for i in range(0, len(audio_bytes), frame_size):
    frame = audio_bytes[i:i+frame_size]
    if len(frame) < frame_size:
        break
    is_speech = vad.is_speech(frame, sample_rate)
    if is_speech:
        segments.append(f"[Speaker {speaker}] frame {i}")
        speaker = 2 if speaker == 1 else 1

print("Diarized (approx.):\n")
for seg in segments[:20]:  # show sample
    print(seg)


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Diarized (approx.):

[Speaker 1] frame 2880
[Speaker 2] frame 3840
[Speaker 1] frame 4800
[Speaker 2] frame 5760
[Speaker 1] frame 6720
[Speaker 2] frame 7680
[Speaker 1] frame 8640
[Speaker 2] frame 9600
[Speaker 1] frame 10560
[Speaker 2] frame 11520
[Speaker 1] frame 12480
[Speaker 2] frame 13440
[Speaker 1] frame 14400
[Speaker 2] frame 15360
[Speaker 1] frame 16320
[Speaker 2] frame 17280
[Speaker 1] frame 18240
[Speaker 2] frame 19200
[Speaker 1] frame 20160
[Speaker 2] frame 21120


In [5]:
diarized_text = ""
speaker = 1

sentences = text.split(".")
for sentence in sentences:
    if sentence.strip():
        diarized_text += f"[Speaker {speaker}] {sentence.strip()}\n"
        speaker = 2 if speaker == 1 else 1

print(diarized_text)

with open("diarized_transcript.txt", "w") as f:
    f.write(diarized_text)


[Speaker 1] Real self-confidence doesn't come from shouting affirmations in the mirror
[Speaker 2] Real self-confidence comes from giving the world irrefutable proof you are who you say you are
[Speaker 1] I fucking love that



In [6]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
summary = summarizer(diarized_text, max_length=200, min_length=60)[0]['summary_text']

print(summary)

with open("summary_diarized.txt", "w") as f:
    f.write(summary)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Your max_length is set to 200, but your input_length is only 58. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=29)


Real self-confidence doesn't come from shouting affirmations in the mirror. Real self- confidence comes from giving the world irrefutable proof you are who you say you are. [Speaker 1] I fucking love that you're a woman. I love that I'm a man.


In [7]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
scores = scorer.score(diarized_text, summary)

scores


{'rouge1': Score(precision=0.7608695652173914, recall=0.8974358974358975, fmeasure=0.8235294117647058),
 'rouge2': Score(precision=0.7333333333333333, recall=0.868421052631579, fmeasure=0.7951807228915663),
 'rougeL': Score(precision=0.7608695652173914, recall=0.8974358974358975, fmeasure=0.8235294117647058)}